# 07 — Data Cleaning

Applies all fixes identified in `reports/data-quality-2026-06-12.md` and saves cleaned datasets to `data/interim/`.

**Checklist from data quality report:**
1. ELO: replace non-breaking spaces (`\xa0`) in team names
2. ELO: drop Moldova row with rating=0
3. Mart Jürisoo: drop 2 duplicate rows
4. Mart Jürisoo: drop 70 null-score rows (future WC 2026 fixtures — prediction targets, not training data)
5. Apply name harmonization across all datasets (canonical = OpenFootball WC 2026 names)
6. Filter to matches relevant for training (post-1990, competitive, teams present at WC 2026)
7. Use `elo_ratings_wc2026.csv` for current pre-tournament team strength
8. Transfermarkt: log-transform market values; aggregate to squad-level

**Output files (`data/interim/`):**
- `matches.parquet` — cleaned match results (training backbone)
- `elo_history.parquet` — cleaned ELO history (for feature lookups)
- `elo_wc2026.parquet` — WC 2026 pre-tournament ELO snapshot (48 teams)
- `squad_values.parquet` — squad-level market value aggregates per team
- `wc2026_fixtures.parquet` — WC 2026 fixtures with canonical team names (prediction targets)

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

RAW = Path("../data/raw")
INTERIM = Path("../data/interim")
INTERIM.mkdir(exist_ok=True)

## Name harmonization table

Canonical names = OpenFootball / Kaggle WC 2026 fixture names.
Each source maps its variant → canonical.

In [2]:
# Maps each source's variant name -> canonical WC 2026 fixture name
# ELO eloratings.csv uses non-breaking spaces; those are normalized first (see load step),
# so the mapping below is post-normalization (regular spaces).
NAME_MAP: dict[str, str] = {
    # ELO / Mart Jurisoo -> canonical
    "United States": "USA",
    "Bosnia and Herzegovina": "Bosnia & Herzegovina",
    "Czechia": "Czech Republic",
    "Democratic Republic of Congo": "DR Congo",
    # Transfermarkt -> canonical
    "Korea, South": "South Korea",
    "Cote d'Ivoire": "Ivory Coast",
    "Bosnia-Herzegovina": "Bosnia & Herzegovina",
    "Curacao": "Curaçao",
    # Iran is consistent across all sources — no mapping needed
}

# All 48 actual WC 2026 teams (no placeholders)
WC2026_TEAMS: set[str] = {
    "Mexico", "South Africa", "South Korea", "Canada", "Qatar", "Switzerland",
    "Brazil", "Morocco", "Haiti", "Scotland", "USA", "Paraguay", "Australia",
    "Germany", "Curaçao", "Ivory Coast", "Ecuador", "Netherlands", "Japan", "Tunisia",
    "Belgium", "Egypt", "Iran", "New Zealand", "Spain", "Cape Verde", "Saudi Arabia",
    "Uruguay", "France", "Senegal", "Norway", "Argentina", "Algeria", "Austria",
    "Jordan", "Portugal", "Uzbekistan", "Colombia", "England", "Croatia", "Ghana",
    "Panama", "Czech Republic", "DR Congo", "Bosnia & Herzegovina", "Iraq",
    "Turkey", "Sweden",
}


def harmonize(series: pd.Series) -> pd.Series:
    """Normalize non-breaking spaces then apply name map."""
    return series.str.replace("\xa0", " ", regex=False).map(lambda x: NAME_MAP.get(x, x))


print(f"Name map entries: {len(NAME_MAP)}")
print(f"WC 2026 teams: {len(WC2026_TEAMS)}")

Name map entries: 8
WC 2026 teams: 48


## 1. Match results — Mart Jürisoo `results.csv`

In [3]:
raw_results = pd.read_csv(RAW / "mart_jurisoo/results.csv", parse_dates=["date"])
print(f"Raw rows: {len(raw_results):,}")

# --- Fix 1: drop null-score rows (future WC 2026 fixtures) ---
mask_no_score = raw_results["home_score"].isna()
print(f"Null-score rows (future fixtures): {mask_no_score.sum()}")
results = raw_results[~mask_no_score].copy()

# --- Fix 2: drop 2 duplicate rows ---
before = len(results)
results = results.drop_duplicates(subset=["date", "home_team", "away_team"], keep="first")
print(f"Duplicates dropped: {before - len(results)}")

# --- Fix 3: harmonize team names ---
results["home_team"] = harmonize(results["home_team"])
results["away_team"] = harmonize(results["away_team"])

# --- Fix 4: cast scores to int ---
results["home_score"] = results["home_score"].astype(int)
results["away_score"] = results["away_score"].astype(int)

print(f"\nCleaned rows: {len(results):,}")
print(f"Date range: {results['date'].min().date()} – {results['date'].max().date()}")
results.head(3)

Raw rows: 49,477
Null-score rows (future fixtures): 70
Duplicates dropped: 2

Cleaned rows: 49,405
Date range: 1872-11-30 – 2026-06-11


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False


In [4]:
# --- Filter: post-1990, drop scoreless anomalies already removed ---
# Rationale: pre-1990 football is a different era (different tactics, data quality lower);
# Dixon-Coles / ELO literature typically trains on 1990+.
# Keep ALL tournaments for now — feature engineering will weight by tournament type.
matches = results[results["date"].dt.year >= 1990].copy()
print(f"Post-1990 rows: {len(matches):,}")

# Verify WC 2026 teams are all covered
all_teams_in_matches = set(matches["home_team"]) | set(matches["away_team"])
missing = WC2026_TEAMS - all_teams_in_matches
print(f"WC 2026 teams missing from match history: {missing}")

matches.to_parquet(INTERIM / "matches.parquet", index=False)
print(f"\nSaved → data/interim/matches.parquet ({len(matches):,} rows)")

Post-1990 rows: 32,288
WC 2026 teams missing from match history: set()

Saved → data/interim/matches.parquet (32,288 rows)


## 2. ELO history — `eloratings.csv`

In [5]:
raw_elo = pd.read_csv(RAW / "elo_ratings/eloratings.csv")
print(f"Raw rows: {len(raw_elo):,}")

# --- Fix 1: parse date — file has mixed formats (YYYY-MM-DD and M/D/YYYY) ---
raw_elo["date"] = pd.to_datetime(raw_elo["date"], format="mixed", dayfirst=False)

# --- Fix 2: normalize non-breaking spaces in team names ---
raw_elo["team"] = raw_elo["team"].str.replace("\xa0", " ", regex=False)

# --- Fix 3: apply name map ---
raw_elo["team"] = raw_elo["team"].map(lambda x: NAME_MAP.get(x, x))

# --- Fix 4: drop Moldova rating=0 anomaly ---
bad_rows = (raw_elo["rating"] == 0) | (raw_elo["rating"].isna())
print(f"Rating=0 or null rows dropped: {bad_rows.sum()}")
elo = raw_elo[~bad_rows].copy()

# --- Filter: post-1990 only (aligns with matches filter) ---
elo = elo[elo["date"].dt.year >= 1990].copy()

print(f"\nCleaned rows: {len(elo):,}")
print(f"Date range: {elo['date'].min().date()} – {elo['date'].max().date()}")
print(f"Unique teams: {elo['team'].nunique()}")

elo.to_parquet(INTERIM / "elo_history.parquet", index=False)
print(f"\nSaved → data/interim/elo_history.parquet ({len(elo):,} rows)")

Raw rows: 6,678
Rating=0 or null rows dropped: 32

Cleaned rows: 5,565
Date range: 1990-03-18 – 2025-12-13
Unique teams: 251

Saved → data/interim/elo_history.parquet (5,565 rows)


## 3. ELO WC 2026 snapshot — `elo_ratings_wc2026.csv`

This is the correct source for pre-tournament team strength (not the stale `eloratings.csv`).

In [6]:
raw_wc = pd.read_csv(RAW / "elo_ratings/elo_ratings_wc2026.csv", parse_dates=["snapshot_date"])
print(f"Raw rows: {len(raw_wc):,} | Snapshots: {raw_wc['snapshot_date'].nunique()}")

# Keep only the most recent snapshot per team (pre-tournament)
# The dataset extends to 2026-12-31 (projected) — use the latest available snapshot
latest_date = raw_wc["snapshot_date"].max()
print(f"Latest snapshot date: {latest_date.date()}")

elo_wc = raw_wc[raw_wc["snapshot_date"] == latest_date].copy()

# Harmonize country names
elo_wc["country"] = elo_wc["country"].map(lambda x: NAME_MAP.get(x, x))

# Keep only the 48 WC 2026 teams
elo_wc = elo_wc[elo_wc["country"].isin(WC2026_TEAMS)].copy()

# Select useful columns
elo_wc = elo_wc[[
    "country", "rank", "rating", "confederation",
    "matches_total", "wins", "losses", "draws",
    "goals_for", "goals_against",
]].rename(columns={"country": "team"})

missing_wc = WC2026_TEAMS - set(elo_wc["team"])
print(f"WC 2026 teams missing from snapshot: {missing_wc}")
print(f"\nRows: {len(elo_wc)}")
print(elo_wc.sort_values("rank").head(10).to_string(index=False))

elo_wc.to_parquet(INTERIM / "elo_wc2026.parquet", index=False)
print(f"\nSaved → data/interim/elo_wc2026.parquet ({len(elo_wc)} rows)")

Raw rows: 4,683 | Snapshots: 127
Latest snapshot date: 2026-12-31
WC 2026 teams missing from snapshot: set()

Rows: 48
       team  rank  rating confederation  matches_total  wins  losses  draws  goals_for  goals_against
      Spain     1    2165          UEFA            780   461     138    181       1591            697
  Argentina     2    2113      CONMEBOL           1109   610     228    271       2112           1136
     France     3    2081          UEFA            938   474     269    195       1706           1272
    England     4    2020          UEFA           1160   683     215    262       2719           1118
     Brazil     5    1984      CONMEBOL           1065   670     172    223       2294            954
   Portugal     5    1984          UEFA            694   346     189    159       1222            779
   Colombia     7    1975      CONMEBOL            648   265     205    178        842            741
Netherlands     8    1961          UEFA            901   465     

## 4. Transfermarkt — squad-level market values

In [7]:
tm_players = pd.read_csv(RAW / "transfermarkt/players.csv")
tm_vals = pd.read_csv(RAW / "transfermarkt/player_valuations.csv", parse_dates=["date"])

print(f"Players: {len(tm_players):,} | Valuations: {len(tm_vals):,}")

# Harmonize citizenship country names
tm_players["country_of_citizenship"] = tm_players["country_of_citizenship"].map(
    lambda x: NAME_MAP.get(str(x), x) if pd.notna(x) else x
)

# Keep only players from WC 2026 nations
wc_players = tm_players[tm_players["country_of_citizenship"].isin(WC2026_TEAMS)].copy()
print(f"\nPlayers from WC 2026 nations: {len(wc_players):,}")

Players: 47,716 | Valuations: 507,815

Players from WC 2026 nations: 29,801


In [8]:
# Get latest valuation per player from the valuations table (more complete than players.csv)
latest_vals = (
    tm_vals.sort_values("date")
    .groupby("player_id", as_index=False)
    .last()[["player_id", "market_value_in_eur", "date"]]
    .rename(columns={"market_value_in_eur": "latest_mv", "date": "mv_date"})
)

print(f"Latest valuations: {len(latest_vals):,}")
print(f"Valuation date range: {latest_vals['mv_date'].min().date()} – {latest_vals['mv_date'].max().date()}")

# Merge onto WC 2026 players
wc_players = wc_players.merge(latest_vals, on="player_id", how="left")

# Use latest_mv from valuations table; fall back to market_value_in_eur from players.csv
wc_players["mv"] = wc_players["latest_mv"].fillna(wc_players["market_value_in_eur"])
wc_players["mv"] = wc_players["mv"].clip(lower=0)  # no negative values

# Log-transform (log1p to handle 0s)
wc_players["log_mv"] = np.log1p(wc_players["mv"])

print(f"\nMV coverage: {wc_players['mv'].notna().mean()*100:.1f}%")

Latest valuations: 31,507
Valuation date range: 2005-12-02 – 2026-02-27

MV coverage: 63.0%


In [9]:
# Aggregate to squad level: sum of top-23 players by market value per nation
# (typical WC squad size; reduces noise from bench/youth players with MV=0)
def squad_agg(group: pd.DataFrame) -> pd.Series:
    top23 = group.nlargest(23, "mv", keep="all").head(23)
    return pd.Series({
        "squad_mv_sum_top23": top23["mv"].sum(),
        "squad_mv_median": group["mv"].median(),
        "squad_mv_mean": group["mv"].mean(),
        "n_players_with_mv": (group["mv"] > 0).sum(),
        "n_players_total": len(group),
        "mv_coverage_pct": (group["mv"] > 0).mean() * 100,
    })

squad_values = (
    wc_players.groupby("country_of_citizenship")
    .apply(squad_agg, include_groups=False)
    .reset_index()
    .rename(columns={"country_of_citizenship": "team"})
)

# Log-transform the aggregated squad value
squad_values["log_squad_mv_top23"] = np.log1p(squad_values["squad_mv_sum_top23"])

print(squad_values.sort_values("squad_mv_sum_top23", ascending=False).head(20).to_string(index=False))

squad_values.to_parquet(INTERIM / "squad_values.parquet", index=False)
print(f"\nSaved → data/interim/squad_values.parquet ({len(squad_values)} teams)")

       team  squad_mv_sum_top23  squad_mv_median  squad_mv_mean  n_players_with_mv  n_players_total  mv_coverage_pct  log_squad_mv_top23
    England        1665000000.0         300000.0   3.237546e+06             1469.0           1682.0        87.336504           21.233091
     France        1630000000.0         300000.0   2.751423e+06             1637.0           1890.0        86.613757           21.211846
     Brazil        1243000000.0         350000.0   2.308337e+06             1615.0           2700.0        59.814815           20.940794
      Spain        1165000000.0         300000.0   2.016875e+06             1952.0           2169.0        89.995390           20.875987
   Portugal        1109000000.0         250000.0   2.140510e+06             1107.0           1261.0        87.787470           20.826725
    Germany        1096000000.0         300000.0   2.082556e+06             1332.0           1452.0        91.735537           20.814933
Netherlands        1024000000.0         2

## 5. WC 2026 fixtures — prediction targets

In [10]:
import json

with open(RAW / "openfootball_2026/worldcup_2026.json") as f:
    wc_json = json.load(f)

# Placeholder names used in knockout rounds before pairings are resolved
PLACEHOLDER_PREFIXES = ("1", "2", "3", "W", "L")


def is_placeholder(name: str) -> bool:
    return any(name.startswith(p) and not name[1:2].isalpha() for p in PLACEHOLDER_PREFIXES) or "/" in name


rows = []
for m in wc_json["matches"]:
    t1, t2 = m["team1"], m["team2"]
    if is_placeholder(t1) or is_placeholder(t2):
        continue  # skip unresolved knockout pairings
    score = m.get("score", {})
    ft = score.get("ft") if score else None
    rows.append({
        "date": pd.to_datetime(m["date"]),
        "home_team": NAME_MAP.get(t1, t1),
        "away_team": NAME_MAP.get(t2, t2),
        "group": m.get("group"),
        "round": m.get("round"),
        "venue": m.get("ground"),
        "home_score": ft[0] if ft else None,
        "away_score": ft[1] if ft else None,
        "played": ft is not None,
    })

fixtures = pd.DataFrame(rows)
print(f"Total resolved fixtures: {len(fixtures)}")
print(f"Played: {fixtures['played'].sum()} | Remaining: {(~fixtures['played']).sum()}")
print(f"\nSample:")
fixtures.head(5)

Total resolved fixtures: 80
Played: 2 | Remaining: 78

Sample:


,date,home_team,away_team,group,round,venue,home_score,away_score,played
0,2026-06-11,Mexico,South Africa,Group A,Matchday 1,Mexico City,2.0,0.0,True
1,2026-06-11,South Korea,Czech Republic,Group A,Matchday 1,Guadalajara (Zapopan),2.0,1.0,True
2,2026-06-18,Czech Republic,South Africa,Group A,Matchday 8,Atlanta,NaN,NaN,False
3,2026-06-18,Mexico,South Korea,Group A,Matchday 8,Guadalajara (Zapopan),NaN,NaN,False
4,2026-06-24,Czech Republic,Mexico,Group A,Matchday 14,Mexico City,NaN,NaN,False


In [11]:
# Verify all fixture teams are in canonical WC2026 set
fixture_teams = set(fixtures["home_team"]) | set(fixtures["away_team"])
unknown = fixture_teams - WC2026_TEAMS
print(f"Unknown team names in fixtures: {unknown}")

fixtures.to_parquet(INTERIM / "wc2026_fixtures.parquet", index=False)
print(f"\nSaved → data/interim/wc2026_fixtures.parquet ({len(fixtures)} rows)")

Unknown team names in fixtures: {'2E', '2H', '2L', '2F', '2I', '2C', '1J', '1F', '2G', '2A', '2K', '1H', '2D', '2B', '2J', '1C'}

Saved → data/interim/wc2026_fixtures.parquet (80 rows)


## Summary

In [12]:
print("=" * 60)
print("DATA CLEANING SUMMARY")
print("=" * 60)

outputs = [
    ("matches.parquet", "Training match results (post-1990, cleaned)"),
    ("elo_history.parquet", "ELO ratings history (post-1990, names fixed)"),
    ("elo_wc2026.parquet", "WC 2026 pre-tournament ELO snapshot (48 teams)"),
    ("squad_values.parquet", "Transfermarkt squad values (top-23 sum per nation)"),
    ("wc2026_fixtures.parquet", "WC 2026 resolved fixtures (prediction targets)"),
]

for fname, desc in outputs:
    path = INTERIM / fname
    if path.exists():
        df = pd.read_parquet(path)
        print(f"  ✓ {fname}: {len(df):,} rows — {desc}")
    else:
        print(f"  ✗ {fname}: MISSING")

DATA CLEANING SUMMARY


  ✓ matches.parquet: 32,288 rows — Training match results (post-1990, cleaned)
  ✓ elo_history.parquet: 5,565 rows — ELO ratings history (post-1990, names fixed)
  ✓ elo_wc2026.parquet: 48 rows — WC 2026 pre-tournament ELO snapshot (48 teams)
  ✓ squad_values.parquet: 48 rows — Transfermarkt squad values (top-23 sum per nation)
  ✓ wc2026_fixtures.parquet: 80 rows — WC 2026 resolved fixtures (prediction targets)
